# 02 - XGB / LR / RF Training (CPU, cumulative accuracy)
Train baseline models locally, track cumulative precision/recall/false positive trends, and export curves to artifacts/model_eval.

## 1) Environment Setup
CPU-only; no Colab. Uses pandas/numpy/sklearn/matplotlib; optional XGBoost if installed.

In [ ]:
import os
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_recall_curve, roc_curve, auc
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

plt.style.use("dark_background")
sns.set_theme(style="darkgrid")

ROOT = Path(".").resolve()
DATA_PATH = Path(os.getenv("FRAUD_DATA_PATH", ROOT / "data" / "processed" / "transactions.parquet"))
ARTIFACT_DIR = ROOT / "artifacts" / "model_eval"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RNG_SEED = 42
np.random.seed(RNG_SEED)

print(f"Using data path: {DATA_PATH}")
print(f"Artifacts → {ARTIFACT_DIR}")

## 2) Load + Feature Engineering
Reuse features from notebook 01; synthesize data if missing.

In [ ]:
def synthesize_transactions(n_rows: int = 50000, fraud_ratio: float = 0.01, rng_seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(rng_seed)
    labels = rng.choice([0, 1], size=n_rows, p=[1 - fraud_ratio, fraud_ratio])
    amount = rng.gamma(shape=2.0, scale=200.0, size=n_rows)
    oldbalanceOrg = rng.normal(loc=5000, scale=1500, size=n_rows)
    newbalanceOrig = oldbalanceOrg - amount * rng.uniform(0.8, 1.0, size=n_rows)
    oldbalanceDest = rng.normal(loc=2000, scale=1000, size=n_rows)
    newbalanceDest = oldbalanceDest + amount * rng.uniform(0.7, 1.0, size=n_rows)
    tx_type = rng.choice(["PAYMENT", "TRANSFER", "CASH_OUT", "DEBIT"], size=n_rows)
    return pd.DataFrame({
        "amount": amount,
        "oldbalanceOrg": oldbalanceOrg,
        "newbalanceOrig": newbalanceOrig,
        "oldbalanceDest": oldbalanceDest,
        "newbalanceDest": newbalanceDest,
        "type": tx_type,
        "is_fraud": labels,
    })


def load_dataset(path: Path) -> pd.DataFrame:
    if path.exists():
        if path.suffix.lower() == ".parquet":
            df = pd.read_parquet(path)
        else:
            df = pd.read_csv(path)
        print(f"Loaded dataset from {path} with shape {df.shape}")
        return df
    print(f"Data path not found: {path}. Synthesizing sample dataset (~1% fraud).")
    return synthesize_transactions()


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["amount_log"] = np.log1p(df["amount"].clip(lower=0))
    df["balance_delta_org"] = df["oldbalanceOrg"] - df["newbalanceOrig"]
    df["balance_delta_dest"] = df["newbalanceDest"] - df["oldbalanceDest"]
    for col in ["amount", "balance_delta_org", "balance_delta_dest"]:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        df[f"outlier_flag_{col}"] = ((df[col] < lower) | (df[col] > upper)).astype(int)
    return df


df_raw = load_dataset(DATA_PATH)
df = add_features(df_raw)

cat_cols = ["type"]
num_cols = [c for c in df.columns if c not in cat_cols + ["is_fraud"]]

numeric_transformer = Pipeline(steps=[("scaler", StandardScaler())])

categorical_transformer = Pipeline(steps=[("encoder", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ]
)

print(f"Numeric cols: {num_cols}")
print(f"Categorical cols: {cat_cols}")

## 3) Train-Test Split
Stratified split to preserve fraud ratio.

In [ ]:
X = df.drop(columns=["is_fraud"])
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RNG_SEED, stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train fraud ratio:", y_train.mean().round(4), "Test fraud ratio:", y_test.mean().round(4))

## 4) Model Training
Lightweight CPU-friendly params; class_weight balanced. Models: LR, RF, optional XGB.

In [ ]:
models = {}

log_reg = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", LogisticRegression(max_iter=400, class_weight="balanced", n_jobs=-1, C=0.5)),
])
models["lr"] = log_reg

rf_clf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=160,
        max_depth=14,
        min_samples_leaf=2,
        n_jobs=-1,
        class_weight="balanced_subsample",
        random_state=RNG_SEED,
    )),
])
models["rf"] = rf_clf

if HAS_XGB:
    xgb_clf = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("clf", XGBClassifier(
            max_depth=6,
            n_estimators=220,
            learning_rate=0.07,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.2,
            tree_method="hist",
            objective="binary:logistic",
            eval_metric="aucpr",
            random_state=RNG_SEED,
        )),
    ])
    models["xgb"] = xgb_clf

trained_models = {}
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    trained_models[name] = model
print("Training complete.")

## 5) Cumulative Accuracy / Precision / Recall / FPR
Compute cumulative metrics by sorting predictions descending on fraud probability (proxy for transaction ranking). Save per-model curves.

In [ ]:
def cumulative_metrics(y_true: np.ndarray, probs: np.ndarray) -> pd.DataFrame:
    order = np.argsort(-probs)
    y_sorted = y_true[order]
    probs_sorted = probs[order]
    cum_tp = np.cumsum(y_sorted)
    cum_fp = np.cumsum(1 - y_sorted)
    idx = np.arange(1, len(y_sorted) + 1)
    precision = cum_tp / idx
    recall = cum_tp / (y_sorted.sum() + 1e-9)
    fpr = cum_fp / (cum_fp[-1] + 1e-9)
    return pd.DataFrame({
        "k": idx,
        "proba": probs_sorted,
        "precision": precision,
        "recall": recall,
        "fpr": fpr,
    })


curve_rows = []

for name, model in trained_models.items():
    probs = model.predict_proba(X_test)[:, 1]
    cm_df = cumulative_metrics(y_test.to_numpy(), probs)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(cm_df["k"], cm_df["precision"], label="precision")
    ax.plot(cm_df["k"], cm_df["recall"], label="recall")
    ax.plot(cm_df["k"], cm_df["fpr"], label="false positive rate")
    ax.set_xlabel("Transactions (sorted by risk)")
    ax.set_ylabel("Metric")
    ax.set_title(f"Cumulative metrics - {name}")
    ax.legend()
    path = ARTIFACT_DIR / f"{name}_accuracy_curve.png"
    fig.savefig(path)
    plt.close(fig)
    print(f"Saved cumulative curve to {path}")

    pr, rc, _ = precision_recall_curve(y_test, probs)
    fpr_curve, tpr_curve, _ = roc_curve(y_test, probs)
    pr_auc = auc(rc, pr)
    roc_auc = auc(fpr_curve, tpr_curve)
    curve_rows.append({
        "model": name,
        "pr_auc": pr_auc,
        "roc_auc": roc_auc,
        "final_precision": cm_df["precision"].iloc[-1],
        "final_recall": cm_df["recall"].iloc[-1],
    })

metrics_table = pd.DataFrame(curve_rows)
metrics_path = ARTIFACT_DIR / "model_eval_summary.csv"
metrics_table.to_csv(metrics_path, index=False)
print(f"Saved metrics table to {metrics_path}")
metrics_table

## 6) Save Curves + Threshold Sweep
Save PR/ROC plus threshold table for the best model (by PR AUC).

In [ ]:
# Pick best by PR AUC
best_model_name = metrics_table.sort_values(by="pr_auc", ascending=False).iloc[0]["model"]
best_model = trained_models[best_model_name]
probs = best_model.predict_proba(X_test)[:, 1]

pr, rc, _ = precision_recall_curve(y_test, probs)
fpr_curve, tpr_curve, _ = roc_curve(y_test, probs)

fig_pr, ax_pr = plt.subplots(figsize=(8, 6))
ax_pr.plot(rc, pr, label=f"{best_model_name} (AUC={auc(rc, pr):.3f})")
ax_pr.set_xlabel("Recall")
ax_pr.set_ylabel("Precision")
ax_pr.set_title("Precision-Recall (best model)")
ax_pr.legend()
pr_path = ARTIFACT_DIR / "best_model_pr_curve.png"
fig_pr.savefig(pr_path)
plt.close(fig_pr)

fig_roc, ax_roc = plt.subplots(figsize=(8, 6))
ax_roc.plot(fpr_curve, tpr_curve, label=f"{best_model_name} (AUC={auc(fpr_curve, tpr_curve):.3f})")
ax_roc.set_xlabel("FPR")
ax_roc.set_ylabel("TPR")
ax_roc.set_title("ROC (best model)")
ax_roc.legend()
roc_path = ARTIFACT_DIR / "best_model_roc_curve.png"
fig_roc.savefig(roc_path)
plt.close(fig_roc)

thresholds = np.linspace(0.1, 0.9, 9)
th_rows = []
for th in thresholds:
    preds = (probs >= th).astype(int)
    tp = ((preds == 1) & (y_test == 1)).sum()
    fp = ((preds == 1) & (y_test == 0)).sum()
    fn = ((preds == 0) & (y_test == 1)).sum()
    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    th_rows.append({"threshold": th, "precision": precision, "recall": recall})

th_df = pd.DataFrame(th_rows)
th_path = ARTIFACT_DIR / "best_model_thresholds.csv"
th_df.to_csv(th_path, index=False)
print(f"Saved best model PR/ROC and threshold table to {pr_path}, {roc_path}, {th_path}")
th_df